In [ ]:
import os
import json
import pandas as pd
import time
import numpy as np

In [ ]:
stations = {
    "Gv": "The Hague",
    "Dt": "Delft",
    "Dtcp": "Delft",
    "Laa": "The Hague",
    "Rtd": "Rotterdam",
    "Shl": "Schiphol",
    "Hfd": "Hoofddorp",
    "Ledn": "Leiden",
    "Sdm": "Schiedam",
    "Rmoa": "Rotterdam",
    "Gvm": "The Hague"
}


### Info scenarios (Appendix)

In [ ]:
num_trains = {}
scen_time = {}
graph_data = {}
for scen in [1,2,3,4]:
	with open(os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__")))), "data", "railways", "case_study_scenarios", f"2025-07-08_{scen}.json"), "r") as f:
		jsonobj = json.load(f)
		num_trains[scen] = len(jsonobj["trains"])
		scen_time[scen] = max([x["movements"]["endTime"] for x in jsonobj["trains"]]) / 60
	with open(os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__")))), "db-results", f"resultsEurostar{scen-1}", f"fsipp_FlexSIPP_Eurostar_graph-{scen-1}.txt"), "r") as f:
		nodes = f.readline().strip().split("vertex count: ")[1]
		edges = f.readline().strip().split("edge count: ")[1]
		graph_data[scen] = (nodes, edges)
	print(f"\t\item Scenario {scen} has {num_trains[scen]} trains in a {scen_time[scen]:.0f}-minute timespan (resulting in {graph_data[scen][0]} safe intervals with {graph_data[scen][1]} ATFs)")

### Tipping points (Table 1)

In [ ]:
scenario_tipping_point = 1
path = os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__")))), "db-results", f"resultsEurostar{scenario_tipping_point-1}")
lines = [r"\begin{table}[t]", r"\centering", r"\caption{Tipping points found by FlexSIPP for Scenario~" + str(scenario_tipping_point) + r"(" + f"{num_trains[scenario_tipping_point]}" + r" trains over a " + f"{scen_time[scenario_tipping_point]:.0f}" + r"-minute timespan). Times in \texttt{mm:ss}.}", r"\label{tab:tip}", r"\begin{tabular}{llll}", r"\toprule", r"Tipping Point & Train & Location & Delay \\", r"\midrule"]
with open(os.path.join(path, f"tipping_points_eurostar-{scenario_tipping_point-1}.json"), "r") as f:
    tipping_points = json.load(f)
    for point in tipping_points:
        print(f"Eurostar should reach {point['location'][0]['loc']} before {point['time']:.2f} by delaying trains:")
        print(' and'.join([train + " at " + " and at ".join([f"{node} for {x:.2f}" for node, x in delay.items()]) for train, delay in point["delays"].items() if delay]))
        for train, delay in point["delays"].items():
            for node, d in delay.items():
                new_line = str(time.strftime('%M:%S', time.gmtime(point["time"]))) + r" & " + str(train) + r" & " + stations[node.split(" ")[-1].split("|")[0]] + r" & " + str(time.strftime('%M:%S', time.gmtime(d))) + r" \\"
                if lines[-1] != new_line:
                    lines.append(new_line)
        # lines.append(r"\midrule")
if lines[-1] == r"\midrule":
    lines.pop(-1)
lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")
lines.append(r"\end{table}")
print()
print("\n".join(lines))

## Read Data

In [ ]:
df = pd.DataFrame(columns=["Route Creation Network", "Conflict Generation", "Flexibility Generation", "Interval Generation", "Search Time FlexSIPP", "Search Time @MAEDeR", "Found Paths FlexSIPP", "Found Paths @MAEDeR", "Nodes Expanded FlexSIPP", "Nodes Expanded @MAEDeR", "Path Lengths mean FlexSIPP", "Path Lengths mean @MAEDeR", "Path Lengths std FlexSIPP", "Path Lengths std @MAEDeR"])
found_paths = {}
for scen in [0,1,2,3]:
    found_paths[scen] = {}
    curdir = os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__")))), "db-results", f"resultsEurostar{scen}")
    if not os.path.isdir(curdir):
        continue
    new_row = {"Route Creation Network": -1, "Conflict Generation": -1, "Interval Generation": -1, "Flexibility Generation": -1, "Search Time FlexSIPP": -1, "Search Time @MAEDeR": -1, "Found Paths FlexSIPP": -1, "Found Paths @MAEDeR": -1}
    # Route creation = block graph + track graph
    with open(os.path.join(curdir, "timing", "TrackGraph.__init__.csv"), "r") as f:
        new_row["Route Creation Network"] = float(f.read().split("\n")[1])
    with open(os.path.join(curdir, "timing", "BlockGraph.__init__.csv"), "r") as f:
        new_row["Route Creation Network"] += float(f.read().split("\n")[1])
    with open(os.path.join(curdir, "timing", "Scenario.process_blocking_time_intervals.csv"), "r") as f:
        new_row["Conflict Generation"] = float(f.read().split("\n")[1])
    with open(os.path.join(curdir, "timing", "Graph.invert_unsafe_intervals.csv"), "r") as f:
        new_row["Interval Generation"] = float(f.read().split("\n")[1])
    with open(os.path.join(curdir, "timing", "Scenario.compute_flexibility.csv"), "r") as f:
        new_row["Flexibility Generation"] = float(f.read().split("\n")[1])
    for alg in ["FlexSIPP", "@MAEDeR"]:
        found_paths[scen][alg] = {}
        with open(os.path.join(curdir, f"fsipp_{alg}_Eurostar_search_output-{scen}.json"), "r") as f:
            results_data = json.load(f)
            path_lengths = {}
            start_times = {}
            for item in results_data["Result"]["payloads"]:
                if item["payload"]: #skip last empty result
                    atf = str(item["edge_atf"]["atf"])
                    found_paths[scen][alg][atf] = {"alpha": int(item["edge_atf"]["atf"][1]), "delta": float(item["edge_atf"]["atf"][3]), "path_length": len([x for x in item["payload"] if "state" in x])}
                    if atf not in path_lengths:
                        path_lengths[atf] = len([x for x in item["payload"] if "state" in x])
            new_row[f"Found Paths {alg}"] = len(path_lengths)
            new_row[f"Path Lengths mean {alg}"] = np.array(list(path_lengths.values())).mean()
            new_row[f"Path Lengths std {alg}"] = np.array(list(path_lengths.values())).std()
            new_row[f"Path Lengths std {alg}"] = np.array(list(path_lengths.values())).std()
            new_row[f"Search Time {alg}"] = int(results_data["MetaData"]["Search Time"]) / 1000
            new_row[f"Nodes Expanded {alg}"] = int(results_data["MetaData"]["Nodes expanded"]) / 1000
    df.loc[scen] = new_row
df

### New format paths table (Table 2)

In [ ]:
lines = [r"\begin{table}[b]", r"\centering", r"\caption{Number of paths found, showing the different starting times~$\earliestStartTime$ and theassociated average length~$\transitTime$ of those path ATFs.}", r"\label{tab:compare}", r"\begin{tabular}{cccccc}", r"\toprule", r" & & \multicolumn{2}{c}{\textbf{FlexSIPP}} & \multicolumn{2}{c}{\textbf{@MAEDeR}}\\", r"Scen & Start & \# Paths & $\transitTime$ & \# Paths & $\transitTime$ \\", r"\midrule"]
path_length_data = {}
for i, scen in enumerate(found_paths):
	path_length_data[scen] = {}
	for j, (alg, data) in enumerate(found_paths[scen].items()):
		alphas = {x['alpha']: [] for atf, x in data.items()}
		for atf, x in data.items():
			# alphas[x['alpha']].append(x['path_length'])
			alphas[x['alpha']].append(x['delta'])
		path_length_data[scen][alg] = alphas
sort_lines = []
for i, (scen, data) in enumerate(path_length_data.items()):
	for alph, lengths in data["@MAEDeR"].items():
		if alph in data["FlexSIPP"]:
			sort_lines.append(fr"{scen+1} & {alph} & {len(data['FlexSIPP'][alph])} & {np.array(data['FlexSIPP'][alph]).mean():.0f} & {len(lengths)} & {np.array(lengths).mean():.0f} \\")
		else:
			sort_lines.append(fr"{scen+1} & {alph} & - & - & {len(lengths)} & {np.array(lengths).mean():.0f} \\")
	for alph, lengths in data["FlexSIPP"].items():
		if alph not in data["@MAEDeR"]:
			sort_lines.append(fr"{scen+1} & {alph} & {len(lengths)} & {np.array(lengths).mean():.0f} & - & - \\ ")
sort_lines.sort(key=lambda x: (int(x.split(r" & ")[0]), float(x.split(r" & ")[1])))
prev_scen = "1"
for line in sort_lines:
    scen = line.split(r" & ")[0]
    if scen != prev_scen:
        lines.append(r"\cmidrule{2-6}")
    lines.append(line)
    prev_scen = scen
lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")
lines.append(r"\end{table}")
print("\n".join(lines))

### Time components (Table 4 - appendix)

In [ ]:
time_table = df.drop(columns=["Found Paths FlexSIPP", "Found Paths @MAEDeR", "Nodes Expanded FlexSIPP", "Nodes Expanded @MAEDeR"])
df_t = time_table.T
df_t.columns = [f"S{i+1}" for i in range(df_t.shape[1])]
latex = df_t.to_latex(
    caption="Table reporting the components of the runtime in seconds of the two algorithms for four different scenarios of replanning the Eurostar.", 
    label="tab:time",
    float_format="%.2f"
    )
latex = latex.replace("Route Creation Network", r"Creation Routes~$\network$").replace("Generation", "gen.").replace("Time ", "")
lines = latex.split("\n")
for i, line in enumerate(lines):
    if "S1" in lines[i]:
        lines[i] = "Time Components" + lines[i]
    if "Conflict gen" in line:
        lines[i] = r" & ".join([x.split(".")[0]if "gen" not in x else x for x in lines[i].split(r" & ")]) + r" \\"
print("\n".join(lines))

### Old paths table

In [ ]:
path_table = df.drop(columns=["Route Creation Network", "Conflict Generation", "Interval Generation", "Flexibility Generation", "Search Time FlexSIPP", "Search Time @MAEDeR", "Nodes Expanded FlexSIPP", "Nodes Expanded @MAEDeR"])
latex = path_table.to_latex(caption="Number of paths found to accommodate Eurostar.", label="tab:compare", position="b")
lines = latex.split("\n")
for i, line in enumerate(lines):
    if "Found Paths" in lines[i]:
        lines[i] = "Scenario" + line.replace("Found Paths ", "")
    elif "&" in line:
        lines[i] = str(int(lines[i].split(r" & ")[0]) + 1) + lines[i][1:]
lines.insert(1, r"\centering")
print("\n".join(lines))
path_table